# Rolling Heuristic Selector v1

QuantBench-inspired continual / distribution-shift experiment for SOXL. This notebook screens a rolling selector over the best heuristic strategy candidates. The selector chooses candidates from past lookback performance only, then evaluates the resulting position path against the hard gate: `MaxDD < 20%` and return above Buy&Hold in each required window.

External reference: arXiv 2504.18600 QuantBench highlights distribution shift, continual learning, ensemble robustness, and task-aligned portfolio metrics.

## Governance

- Direction: `model_selection`
- Role: diagnose / rolling selector baseline
- Changed layer: selection over strategy candidates, not PPO training
- Baseline: `heuristic_strategy_search_v1` found zero passing static configs
- Hypothesis: rolling selection can preserve the 2022 defensive candidate while switching to better participation candidates for 2023-2024
- Acceptance: rolling selector passes both `test_2022` and `test_2023_2024` hard gates
- Failure mode: underlying heuristic candidate family lacks any strong 2023-2024 positive-excess branch.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "SPEC.md").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CARD = ROOT / "runtime/model_selection/card/rolling_heuristic_selector_v1"
CARD.mkdir(parents=True, exist_ok=True)
print(ROOT)

/Users/rziiiii/.openclaw/workspace


In [2]:
import pandas as pd
from src.soxl_quant.app.heuristic_strategy_search import (
    DEFAULT_HEURISTIC_WINDOWS,
    build_heuristic_position_path,
    default_heuristic_config_grid,
    rolling_select_candidate_path,
    screen_heuristic_strategies,
)
from src.soxl_quant.app.quantbench_gates import compute_backtest_metrics, slice_window

panel = pd.read_csv(ROOT / "data/processed/panels/soxl_soxx_vix_full.csv")
configs = default_heuristic_config_grid()
screen = screen_heuristic_strategies(panel, configs, max_drawdown_pct=20.0)
screen.to_csv(CARD / "heuristic_strategy_screen.csv", index=False)
screen.head(12)

,config_id,gate_pass,score,momentum_lookback,soxx_momentum_lookback,require_soxx_confirmation,vix_threshold,bull_position,risk_position,cash_position,...,test_2022_max_drawdown_pct,test_2022_gate_pass,test_2022_failures,test_2023_2024_rows,test_2023_2024_strategy_return_pct,test_2023_2024_buyhold_return_pct,test_2023_2024_relative_return_pct,test_2023_2024_max_drawdown_pct,test_2023_2024_gate_pass,test_2023_2024_failures
0,236,False,493.104201,20,60,True,18.0,1.00,0.0,0.0,...,17.241801,True,,502,171.071669,191.773509,-20.701840,28.033556,False,"max_drawdown_ge_20,relative_return_not_positive"
1,239,False,500.799618,20,60,True,18.0,1.00,0.2,0.0,...,24.215522,False,max_drawdown_ge_20,502,173.231947,191.773509,-18.541563,31.799976,False,"max_drawdown_ge_20,relative_return_not_positive"
2,242,False,727.997243,20,60,True,18.0,1.00,0.4,0.0,...,30.891865,False,max_drawdown_ge_20,502,172.533312,191.773509,-19.240197,35.465424,False,"max_drawdown_ge_20,relative_return_not_positive"
3,230,False,2429.042643,20,60,True,18.0,0.85,0.2,0.0,...,21.951653,False,max_drawdown_ge_20,502,143.183837,191.773509,-48.589672,28.017323,False,"max_drawdown_ge_20,relative_return_not_positive"
4,227,False,2567.838010,20,60,True,18.0,0.85,0.0,0.0,...,14.769610,True,,502,141.261132,191.773509,-50.512377,24.042002,False,"max_drawdown_ge_20,relative_return_not_positive"
5,233,False,2640.971663,20,60,True,18.0,0.85,0.4,0.0,...,28.827435,False,max_drawdown_ge_20,502,142.562034,191.773509,-49.211476,31.886071,False,"max_drawdown_ge_20,relative_return_not_positive"
6,290,False,2907.489809,20,60,True,26.0,1.00,0.0,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"
7,293,False,2907.489809,20,60,True,26.0,1.00,0.2,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"
8,296,False,2907.489809,20,60,True,26.0,1.00,0.4,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"
9,317,False,2907.489809,20,60,True,32.0,1.00,0.0,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"


In [3]:
top_n = 12
selected_config_ids = [int(v) for v in screen.head(top_n)["config_id"].tolist()]
selected_configs = [configs[idx] for idx in selected_config_ids]
candidate_paths = [build_heuristic_position_path(panel, config) for config in selected_configs]
rolling_daily = rolling_select_candidate_path(
    panel,
    candidate_paths,
    lookback_days=252,
    rebalance_days=21,
    min_history_days=252,
    max_drawdown_pct=20.0,
    initial_candidate_id=0,
)
rolling_daily.to_csv(CARD / "rolling_selector_daily.csv", index=False)
rolling_daily.tail()

,datetime,price,strategy_nav,buyhold_nav,position,selected_candidate_id
4079,2026-05-29,224.339996,11.897609,339.651779,1.0,6
4080,2026-06-01,227.029999,12.040270,343.724455,1.0,6
4081,2026-06-02,266.320007,14.123970,403.209707,1.0,6
4082,2026-06-03,280.540009,14.878111,424.738854,1.0,6
4083,2026-06-04,262.700012,13.931988,397.729018,1.0,6


In [4]:
from dataclasses import asdict

metric_rows = []
gate_pass = True
for window_name, (start, end) in DEFAULT_HEURISTIC_WINDOWS.items():
    metric = compute_backtest_metrics(
        slice_window(rolling_daily, start, end),
        window=window_name,
        profile="rolling_heuristic_selector",
        max_drawdown_pct=20.0,
    )
    payload = asdict(metric)
    payload["failures"] = list(metric.failures)
    metric_rows.append(payload)
    gate_pass = gate_pass and metric.gate_pass

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(CARD / "rolling_selector_metrics.csv", index=False)
candidate_table = pd.DataFrame([
    {"selector_candidate_id": i, "source_config_id": selected_config_ids[i], **asdict(config)}
    for i, config in enumerate(selected_configs)
])
candidate_table.to_csv(CARD / "rolling_selector_candidates.csv", index=False)
metrics

,profile,path,window,rows,start,end,strategy_return_pct,buyhold_return_pct,relative_return_pct,max_drawdown_pct,gate_pass,failures
0,rolling_heuristic_selector,,test_2022,251,2022-01-03,2022-12-30,-19.134734,-86.588072,67.453338,19.134734,True,[]
1,rolling_heuristic_selector,,test_2023_2024,502,2023-01-03,2024-12-31,171.459874,191.773509,-20.313636,26.405222,False,"[max_drawdown_ge_20, relative_return_not_posit..."


In [5]:
summary = {
    "experiment": "rolling_heuristic_selector_v1",
    "decision": "hard_gate_pass_found" if gate_pass else "hard_gate_no_rolling_selector_passes",
    "promotion_allowed": bool(gate_pass),
    "training_allowed_from_this_card": False,
    "configs_screened": int(len(screen)),
    "top_n": top_n,
    "selected_config_ids": selected_config_ids,
    "lookback_days": 252,
    "rebalance_days": 21,
    "min_history_days": 252,
    "initial_candidate_id": 0,
    "max_drawdown_pct": 20.0,
    "window_metrics": metric_rows,
    "next_action": "The rolling selector passed 2022 but failed 2023-2024 because the static heuristic family has no positive-excess 2023-2024 candidate; next branch should use model replay/BC candidate paths with cached lookback scoring or train a stronger reentry learner.",
    "sidecars": [
        "summary.json",
        "heuristic_strategy_screen.csv",
        "rolling_selector_candidates.csv",
        "rolling_selector_daily.csv",
        "rolling_selector_metrics.csv",
    ],
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
summary

{'experiment': 'rolling_heuristic_selector_v1',
 'decision': 'hard_gate_no_rolling_selector_passes',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'configs_screened': 2592,
 'top_n': 12,
 'selected_config_ids': [236,
  239,
  242,
  230,
  227,
  233,
  290,
  293,
  296,
  317,
  320,
  323],
 'lookback_days': 252,
 'rebalance_days': 21,
 'min_history_days': 252,
 'initial_candidate_id': 0,
 'max_drawdown_pct': 20.0,
 'window_metrics': [{'profile': 'rolling_heuristic_selector',
   'path': '',
   'window': 'test_2022',
   'rows': 251,
   'start': '2022-01-03',
   'end': '2022-12-30',
   'strategy_return_pct': -19.134734,
   'buyhold_return_pct': -86.588072,
   'relative_return_pct': 67.453338,
   'max_drawdown_pct': 19.134734,
   'gate_pass': True,
   'failures': []},
  {'profile': 'rolling_heuristic_selector',
   'path': '',
   'window': 'test_2023_2024',
   'rows': 502,
   'start': '2023-01-03',
   'end': '2024-12-31',
   'strategy_return_pct': 171.459874,


## Conclusion

The rolling selector mechanism is now implemented and test-covered for causal lookback-only selection. On the default SOXL heuristic candidate family it still does **not** pass the hard gate. The selected path passed `test_2022` with MaxDD below 20%, but failed `test_2023_2024` because return remained below Buy&Hold and MaxDD stayed above 20%.

Decision: do not promote this selector over the current heuristic family. The next useful branch is not another threshold tweak; it is either cached rolling selection over richer model/BC replay candidates or a stronger reentry learner that can create a 2023-2024 positive-excess branch.